In [3]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import requests
import re
import os
import pandas as pd

# =======================================================
# 1. 브라우저 옵션 설정 (이게 중요합니다!)
# =======================================================
chrome_options = Options()

# [중요] 브라우저 꺼짐 방지 (코드 끝나도 창 유지)
chrome_options.add_experimental_option("detach", True)

# [중요] '자동화된 소프트웨어입니다' 문구 제거 (봇 탐지 회피)
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])

# [선택] 화면 최대화 (요소가 안 보이는 에러 방지)
chrome_options.add_argument("--start-maximized")

# [핵심] 사람인 척하기 (User-Agent 설정)
# (이게 없으면 일부 사이트는 접속을 막습니다)
# chrome_options.add_argument("user-agent=Chrome/120.0.0.0")

# =======================================================
# 2. 브라우저 실행 (셀레니움 파트)
# =======================================================
driver = webdriver.Chrome(options=chrome_options)

# 원하는 사이트 접속
url = "https://www.ytn.co.kr"  # 여기만 바꾸면 됩니다
driver.get(url)

# 로딩 기다려주기 (네트워크 느릴 때 대비, 최대 10초 대기)
driver.implicitly_wait(10) 





In [4]:
title_elements = driver.find_elements(By.CLASS_NAME, "news_list")

df = pd.DataFrame([
    {
        '제목': t.text.split('\n')[0].strip(),
        '링크': t.find_element(By.TAG_NAME, "a").get_attribute("href"),
        # ▼ 여기가 핵심 트릭 (사진 없어도 안 멈춤)
        '사진': (t.find_elements(By.TAG_NAME, "img") or [type('dummy', (object,), {'get_attribute': lambda self, x: "없음"})()])[0].get_attribute("src")
    }
    for t in title_elements
])

In [7]:
df = df.drop(0,axis=0)

In [8]:
df

,제목,링크,사진
1,"""8년 만의 메달 도전""...평창 신화 이어갈 세계 3위 ’팀5G’ 출격",https://www.ytn.co.kr/_ln/0107_202602111534020594,https://www.ytn.co.kr/img/news/default_0107.jpg
2,[2PM] 아쉬웠던 쇼트트랙...차준환의 '다 던진 점프',https://www.ytn.co.kr/_ln/0107_202602111512432422,https://image.ytn.co.kr/general/jpg/2026/0211/...
3,"아직 손목에 상처...유승은 ""그만해야지, 했어요"" [앵커리포트]",https://www.ytn.co.kr/_ln/0107_202602111508282282,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
4,"선수촌 식당 가보니...""평창 때는 살 쪄서 나갔는데"" [앵커리포트]",https://www.ytn.co.kr/_ln/0107_202602111448084980,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
5,"김길리 선수 충돌하자 ’100달러’ 쥐고 항의한 코치, 이유는? [앵커리포트]",https://www.ytn.co.kr/_ln/0107_202602111424015391,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
6,"김길리와 충돌한 美 선수, 한국인들 악플 테러에 댓글창 닫아",https://www.ytn.co.kr/_ln/0107_202602111252021070,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
7,쇼트트랙 코치가 100달러 쥐고 뛴 이유는?...밀라노 이모저모 [앵커리포트],https://www.ytn.co.kr/_ln/0107_202602111243433963,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
8,"피겨 차준환, 이번에도 역전 메달?...최가온 출격",https://www.ytn.co.kr/_ln/0107_202602111101345972,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
9,"최가온, 오늘은 내가 날아오를 차례",https://www.ytn.co.kr/_ln/0107_202602111005292071,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."
10,"차준환, 무결점 연기로 프리 진출...괴물들과의 경쟁",https://www.ytn.co.kr/_ln/0107_202602111003485478,"data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP//..."


In [9]:
df.to_csv('news_ytn.csv', index=False)